In [1]:
import json 
import numpy as np
import os
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

ModuleNotFoundError: No module named 'ortools'

In [ ]:
 # --- CVRP PARAMETERS ---
 NUM_VEHICLES     = 4
 VEHICLE_CAPACITY = 100   # units per vehicle
 DEPOT_INDEX      = 0     # Whitefield
 SCALE            = 100   # OR-Tools needs integers - multiply minutes x 100
 SOLVER_TIME_LIMIT = 30   # seconds per instance

In [ ]:
# Generate a CVRP instance with random demands.
def create_cvrp_instance(n_customers=15, seed=42):
    np.random.seed(seed)
    demands = [0] + [int(np.random.randint(10, 31)) for _ in range(n_customers)]
    return {"n_nodes": n_customers + 1, "demands": demands}

In [ ]:
# Solve CVRP using Google OR-Tools.
def solve_cvrp(cost_matrix, instance, label=""):
    n   = instance["n_nodes"]
    dem = instance["demands"]
    
    # Scale travel times to integers
    ci = [[int(cost_matrix[i][j] * SCALE) for j in range(n)] for i in range(n)]
    
    # Create routing model
    manager = pywrapcp.RoutingIndexManager(n, NUM_VEHICLES, DEPOT_INDEX)
    routing = pywrapcp.RoutingModel(manager)
    
    # Cost callback — uses Q10, Q50, or Q90 travel times
    def cost_callback(from_idx, to_idx):
        i = manager.IndexToNode(from_idx)
        j = manager.IndexToNode(to_idx)
        return ci[i][j]
        
    # Demand callback for capacity dimension
    def demand_callback(from_idx):
        return dem[manager.IndexToNode(from_idx)]
        
    cost_cb_idx   = routing.RegisterTransitCallback(cost_callback)
    demand_cb_idx = routing.RegisterUnaryTransitCallback(demand_callback)
    
    # Set arc cost
    routing.SetArcCostEvaluatorOfAllVehicles(cost_cb_idx)

    # Capacity dimension
    routing.AddDimensionWithVehicleCapacity(
        demand_cb_idx,
        0,                                  # no slack
        [VEHICLE_CAPACITY] * NUM_VEHICLES,  # vehicle capacities
        True,                               # start cumul at zero
        "Capacity"
    )
    
    # Allow dropping customers with high penalty (handles infeasible cases)
    penalty = 100000
    for node in range(1, n):
        routing.AddDisjunction([manager.NodeToIndex(node)], penalty)
        
    # Solver settings — Guided Local Search
    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_params.time_limit.seconds = SOLVER_TIME_LIMIT
    search_params.log_search = False
    
    solution = routing.SolveWithParameters(search_params)
    if not solution:
        return {"status":"INFEASIBLE","label":label,"routes":[],"total_cost":9999,"n_vehicles_used":0}

    # Extract routes from solution
    routes = []
    total_cost = 0
    for v in range(NUM_VEHICLES):
        route = []
        idx = routing.Start(v)
        rc = 0
        while not routing.IsEnd(idx):
            node = manager.IndexToNode(idx)
            route.append(node)
            next_idx = solution.Value(routing.NextVar(idx))
            rc += ci[manager.IndexToNode(idx)][manager.IndexToNode(next_idx)]
            idx = next_idx
        route.append(manager.IndexToNode(idx))  # return to depot
        if len(route) > 2:                      # non-empty route
            load = sum(dem[nd] for nd in route if nd != DEPOT_INDEX)
            routes.append({
                "vehicle": v + 1,
                "route":   route,
                "cost_min": round(rc / SCALE, 2),
                "load":    load
            })
            total_cost += rc
            
    return {"status":"FEASIBLE","label":label,"routes":routes,"total_cost":round(total_cost/SCALE,2),"n_vehicles_used":len(routes)}

In [ ]:
# Print routes with zone names.
def print_routes(result, zones):
    print(f"  Strategy: {result['label']}  |  Total: {result['total_cost']} min")
    for r in result["routes"]:
        names = [zones[nd] for nd in r["route"]]
        print(f"  V{r['vehicle']}: {chr(32)+chr(62)+chr(32).join(names)}")
        print(f"           Load: {r['load']}/{VEHICLE_CAPACITY}  Cost: {r['cost_min']} min")

In [ ]:
def run_solver():
    print("SOLVING CVRP — THREE QUANTILE STRATEGIES")
    
    # Load travel time matrices
    with open("travel_time_matrices.json") as f: matrices = json.load(f)
    zones   = matrices["zones"]
    q10     = matrices["Q10"]
    q50     = matrices["Q50"]
    q90     = matrices["Q90"]
    hod     = matrices["hod"]
    print(f"  Hour: {hod}:00 | Zones: {len(zones)} | Vehicles: {NUM_VEHICLES}")
    
    all_q10 = []; all_q50 = []; all_q90 = []
    
    print(f"\n  Running 20 instances per strategy...")
    for seed in range(20):
        inst = create_cvrp_instance(n_customers=15, seed=seed)
        
        r10 = solve_cvrp(q10, inst, f"Q10 seed={seed}")
        r50 = solve_cvrp(q50, inst, f"Q50 seed={seed}")
        r90 = solve_cvrp(q90, inst, f"Q90 seed={seed}")
        
        all_q10.append(r10); all_q50.append(r50); all_q90.append(r90)

        # Print first instance in detail
        if seed == 0:
            print("\n  --- Instance 1 (seed=0) detailed routes ---")
            print_routes(r10, zones)
            print(); print_routes(r50, zones)
            print(); print_routes(r90, zones)

    # Aggregate results
    c10 = [r["total_cost"] for r in all_q10 if r["status"]=="FEASIBLE"]
    c50 = [r["total_cost"] for r in all_q50 if r["status"]=="FEASIBLE"]
    c90 = [r["total_cost"] for r in all_q90 if r["status"]=="FEASIBLE"]
    v10 = [r["n_vehicles_used"] for r in all_q10 if r["status"]=="FEASIBLE"]
    v50 = [r["n_vehicles_used"] for r in all_q50 if r["status"]=="FEASIBLE"]
    v90 = [r["n_vehicles_used"] for r in all_q90 if r["status"]=="FEASIBLE"]
    
    print(f"\n  Summary across 20 instances:")
    print(f"  Q10 avg cost: {np.mean(c10):.1f} min | vehicles: {np.mean(v10):.1f}")
    print(f"  Q50 avg cost: {np.mean(c50):.1f} min | vehicles: {np.mean(v50):.1f}")
    print(f"  Q90 avg cost: {np.mean(c90):.1f} min | vehicles: {np.mean(v90):.1f}")
    gap = (np.mean(c90)-np.mean(c10))/np.mean(c10)*100
    print(f"  Q90 vs Q10 cost gap: {gap:+.1f}%")
    
    output = {
        "hod": hod, "n_instances": 20,
        "q10_results": all_q10, "q50_results": all_q50, "q90_results": all_q90,
        "summary": {
            "q10_avg_cost": round(np.mean(c10), 2),
            "q50_avg_cost": round(np.mean(c50), 2),
            "q90_avg_cost": round(np.mean(c90), 2),
            "q10_avg_vehicles": round(np.mean(v10), 2),
            "q90_avg_vehicles": round(np.mean(v90), 2),
            "q90_vs_q10_pct": round(gap, 2)
        }
    }    
    with open("cvrp_routes.json", "w") as f: json.dump(output, f, indent=2)
    print(f"  Saved: {"cvrp_routes.json"}")
    return output

In [ ]:
run_solver()